# Challenge 3: evaluación final e inferencia

El modelo ya fue seleccionado. Ahora debes abrir el conjunto de test una sola vez, comunicar su desempeño y comprobar que el pipeline puede reutilizarse con datos nuevos.

**Entradas:** `test.csv`, `data_contract.json` y `champion_model.joblib`  
**Salidas:** `test_metrics.json`, `batch_predictions.csv`

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga del modelo y del conjunto de prueba

In [ ]:
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))
training_metadata = json.loads((ARTIFACTS_DIR / "training_metadata.json").read_text(encoding="utf-8"))
model = joblib.load(ARTIFACTS_DIR / "champion_model.joblib")

test_df = pd.read_csv(PROCESSED_DATA_DIR / "test.csv")
X_test = test_df[contract["features"]]
y_test = test_df[contract["target"]]

print("Modelo:", training_metadata["champion"])
print("Test:", X_test.shape)

## 2. Evaluación final

In [ ]:
# Evaluación final del modelo campeón (se abre test UNA sola vez)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy test : {accuracy:.4f}")
print(f"F1 macro test : {f1_macro:.4f}")
print("\nReporte por clase:\n")
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Matriz de confusión — conjunto de test")
plt.tight_layout()
plt.show()

**Análisis:**

> ¿El desempeño es coherente con validación cruzada? ¿Qué clases se confunden? ¿Hay evidencia de sobreajuste?

## 3. Función de inferencia

In [ ]:
feature_columns = contract["features"]

def predict(records: pd.DataFrame) -> pd.DataFrame:
    """Valida el esquema y devuelve las predicciones del pipeline."""
    # 1. verifica columnas faltantes (contrato de datos)
    missing = set(feature_columns) - set(records.columns)
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {sorted(missing)}")

    # 2. selecciona y ORDENA las columnas como en el entrenamiento
    #    (ignora columnas extra; el orden importa para el pipeline)
    X = records[feature_columns].copy()

    # 3. predicción + probabilidades por clase
    preds = model.predict(X)
    output = pd.DataFrame({"prediction": preds}, index=records.index)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        for i, clase in enumerate(model.classes_):
            output[f"proba_{clase}"] = proba[:, i]

    return output

## 4. Predicción por lotes

In [ ]:
# Simulación de datos nuevos. En producción no provendrían del test set.
new_records = X_test.iloc[:5].copy()

batch = predict(new_records)
batch.to_csv(REPORTS_DIR / "batch_predictions.csv", index=False)

print("Predicciones por lotes:")
batch

## 5. Reporte final

In [ ]:
test_metrics = {
    "champion": training_metadata["champion"],
    "cv_accuracy": training_metadata.get("cv_accuracy"),
    "test_accuracy": round(float(accuracy), 4),
    "test_f1_macro": round(float(f1_macro), 4),
    "n_test_samples": int(len(y_test)),
}

(REPORTS_DIR / "test_metrics.json").write_text(
    json.dumps(test_metrics, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(json.dumps(test_metrics, indent=2, ensure_ascii=False))

In [ ]:
assert (REPORTS_DIR / "test_metrics.json").exists()
assert (REPORTS_DIR / "batch_predictions.csv").exists()
print("Challenge 3 completado.")

## Resumen ejecutivo

| Elemento | Resultado |
|---|---|
| Modelo campeón | |
| Accuracy CV | |
| Accuracy test | |
| F1 macro test | |
| Clases más confundidas | |
| Principal limitación | |

## Checklist

- [ ] Evalué únicamente el modelo campeón.
- [ ] No reajusté hiperparámetros después de observar test.
- [ ] Implementé validación básica del esquema de entrada.
- [ ] Generé predicciones por lotes.
- [ ] Guardé las métricas finales.